# Proyecto Loti Perú — Pipeline MLOps
> Notebooks listos para Databricks. Ajustá la variable `DATA_PATH` para tu ruta.

## 02 · Feature Engineering (Silver → Features)

In [0]:
CATALOG = "main"
SCHEMA  = "loterias_silver"
TABLE_SILVER   = f"{CATALOG}.{SCHEMA}.apuestas_silver"
TABLE_FEATURES = f"{CATALOG}.{SCHEMA}.features_apuestas"

from pyspark.sql import functions as F, Window

df = spark.table(TABLE_SILVER)

# Features
w_user = Window.partitionBy("user_id")
w_ip   = Window.partitionBy("ip")
w_user_time = Window.partitionBy("user_id").orderBy(F.col("fecha").cast("timestamp")).rangeBetween(-7*86400, 0)

df_feat = (df
    .withColumn("monto_log", F.log1p("monto"))
    .withColumn("freq_usuario", F.count("*").over(w_user))
    .withColumn("urgencia", F.when(F.col("min_antes_cierre") <= 10, 1).otherwise(0))
    .withColumn("repeticion_ip", F.count("*").over(w_ip))
    .withColumn("prom_monto_usuario", F.avg("monto").over(w_user))
    .withColumn("apuestas_7d_usuario", F.count("tx_id").over(w_user_time))
    .withColumn("tasa_riesgo_ip", F.avg(F.col("es_fraude").cast("double")).over(w_ip))
)

(df_feat.write.format("delta").mode("overwrite").option("overwriteSchema","true")
 .saveAsTable(TABLE_FEATURES))

display(spark.table(TABLE_FEATURES).limit(10))
print("Features OK:", TABLE_FEATURES)